[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/21_mlp_solution.ipynb)

# 🟡 Solution: SwiGLU MLP

*Core Ops & Layers · Medium*

Reference implementation. Try it yourself in `21_mlp.ipynb` first.

---
Implement the **position-wise feed-forward network** — the other half of every
transformer block — as an `nnx.Module`, built from your own weight matrices.

$$\text{MLP}(x) = \text{GELU}(x W_1 + b_1)\, W_2 + b_2$$

with $W_1 \in \mathbb{R}^{d_{in} \times d_{hidden}}$ and
$W_2 \in \mathbb{R}^{d_{hidden} \times d_{out}}$.

### Rules
- Subclass `nnx.Module`; do **not** use `nnx.Linear` or `nnx.MLP` — write the
  matmuls yourself
- Signature: `MLP(din, dout, *, hidden=None, rngs)`
- `hidden=None` must default to **`4 * din`** (the transformer convention)
- Four `nnx.Param` attributes named exactly `w1`, `b1`, `w2`, `b2`
- Each matrix initialised as `normal / sqrt(fan_in)`; both biases zero
- Each matrix drawn from a **separate** key (`rngs.params()` twice)
- `__call__(x)` maps `(..., din) -> (..., dout)` for any number of leading axes
- Either GELU form is accepted (exact erf or the tanh approximation)

### The 4x expansion ratio
The hidden width is 4x the model width in GPT-2, GPT-3, BERT, and ViT. That is a
capacity choice, not a law — but it decides where your parameters live:

| Block component | Parameters (d = model width) |
|---|---|
| Attention (Q, K, V, O) | $4d^2$ |
| MLP ($W_1$ + $W_2$ at 4x) | $8d^2$ |

So **two thirds of a transformer's non-embedding parameters sit in the MLPs**,
not in attention. It is also where the activation memory peaks: the intermediate
tensor is `(B, T, 4d)`, four times the size of the residual stream, which is why
FFN blocks are the first thing people rematerialise (`jax.checkpoint`).

### Where the params live in NNX
Assigning `self.w1 = nnx.Param(...)` in `__init__` is the whole registration
story — NNX walks your attributes, so there is no `register_parameter` step and
no separate params dict threaded through calls. `nnx.split(module)` later pulls
those arrays out into a pytree for `jit`/`grad`, and `nnx.merge` puts them back.

One JAX-specific trap: `jax.nn.gelu` defaults to `approximate=True` (the tanh
form), the opposite of most other frameworks' default. If you are porting
published weights, that mismatch is a silent ~1e-3 error in every activation.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MLP(nnx.Module):
    def __init__(self, din: int, dout: int, *, hidden: int | None = None, rngs: nnx.Rngs):
        if hidden is None:
            hidden = 4 * din                      # transformer expansion ratio

        # Two separate keys: reusing one would make w1 and w2 correlated.
        k1 = rngs.params()
        k2 = rngs.params()

        self.w1 = nnx.Param(jax.random.normal(k1, (din, hidden)) / jnp.sqrt(din))
        self.b1 = nnx.Param(jnp.zeros((hidden,)))
        self.w2 = nnx.Param(jax.random.normal(k2, (hidden, dout)) / jnp.sqrt(hidden))
        self.b2 = nnx.Param(jnp.zeros((dout,)))

        self.din, self.dout, self.hidden = din, dout, hidden

    def __call__(self, x):
        h = x @ self.w1 + self.b1                 # (..., hidden)
        h = jax.nn.gelu(h)
        return h @ self.w2 + self.b2              # (..., dout)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

mlp = MLP(64, 64, rngs=nnx.Rngs(params=0))
print("hidden width:", mlp.w1.shape[1], "= 4 x", mlp.w1.shape[0])

x = jnp.ones((2, 5, 64))
print("(B, T, D):", x.shape, "->", mlp(x).shape)
print("intermediate is 4x wider:", (x @ mlp.w1).shape)

n = sum(int(p.size) for p in jax.tree.leaves(nnx.state(mlp, nnx.Param)))
print("params:", n, " ~8*d^2 =", 8 * 64 ** 2)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("mlp")